# Première inspection d’un dataset

**Projet / problématique commune :** Projet compagnon immo

**Dataset :** BV2022_au_01-01-2024.xlsx

**Responsable :** .. · **Relecteur :** …

**Contributeurs :** Gabrielle RICHERT · **Date :** 14/10/2026

**Version des données :** …

**Référence du fichier / source :** …

## Mode d’emploi collectif

Dupliquer ce notebook pour chaque dataset. Suivre les étapes dans l’ordre et revenir sur une conclusion si une observation ultérieure la remet en cause. Les cellules de code sont vides : vous choisissez les opérations, et pouvez ajouter des cellules.

- Renseigner « non applicable » ou « information inconnue » plutôt que forcer une réponse.
- Conserver les résultats utiles et les figures près de leur commentaire.
- Distinguer observations, hypothèses et décisions ; des points peuvent rester à approfondir.
- Rédiger un seul bilan par grande étape, réutilisable dans le rapport de recherche.
- Faire relire les conclusions par un autre membre du groupe.

**Référence du notebook / script destiné à la remise séparée du code :** …

# A. Bassins de vies entre 2012 et 2022

## 1. Situer la source et ouvrir les données

**À faire dans l’ordre :**
1. Identifier la source et consulter la documentation.
2. Préciser la collecte et le périmètre annoncés.
3. Charger les données et vérifier leur lecture.

| Élément | Information | Référence / réserve |
|---|---|---|
| Producteur / origine |INSEE | aucun fichier de métadonnées associé mais 2 feuilles descriptives dans l'excel|
| Finalité initiale | | |
| Mode de collecte | INSEE, code officiel géographique| |
| Extraction / mise à jour | 2014 à 2022 | |
| Documentation / licence | aucune information| |
| Population / objets annoncés | 1e feuille : nom de chaque bassin de vie, intitulé, type et nombre de communes présentes ; 2e feuilles : par code géographique officiel des communes on trouve son libellé, le bassin de vie auquel elle appartient, le type de commune, son département et sa région | |
| Période / territoire, si pertinents | France avec ses DOM-TOM| |

In [ ]:
bv2012=pd.read_excel('BV2012_2014_2022.xlsx', sheet_name='BV2012', skiprows=5)
compo_communale2012=pd.read_excel('BV2012_2014_2022.xlsx', sheet_name='Composition_communale', skiprows=5)

In [ ]:
# Nombre d'annonces par département 
print('nombres de communes par département :')
print(compo_communale2012["DEP"].value_counts())

print('nombre de communes totales de 2012 à 2022')
print(compo_communale2012["DEP"].value_counts().sum())

In [ ]:
codgeo_bv2012=compo_communale2012.merge(bv2012, on=["BV2012", 'LIBBV2012'],how="left")
codgeo_bv2012.set_index('CODGEO', inplace=True)

In [ ]:
codgeo_bv2012.head()

In [ ]:
cd C:\Users\gabi9\Documents\GITHUB_PROJET_LIORA\projet_compagnon\data\dataset_cible

In [ ]:
codgeo_bv2012.to_parquet('codgeo_bv2012.parquet', index=True)
codgeo_bv2012.to_csv('codgeo_bv2012.csv', index=True)

**Constat :**

Les codes géographiques officiels de chaque commune sont uniques, ils serviront d'index.
On retrouve le détail pour chacun des bassins de vie dans une autre feuille du tableur excel avec les informations le décrivant.

**Impact projet :**

Un fichier unique pour les codes géographique et les bases de vie est possible et essentiel pour la simplification de son utilisation.

**Action envisagée :**

Fusion des informations contenues dans les deux feuilles excel dans un unique DataFrame.

**Incertitudes / limites, si nécessaire :**

Un code géographique peut aussi être un code de bassin de vie. Viligance.

DONNEES DE 2012 A 2022

## 2. Comprendre la structure et une ligne

**À faire dans l’ordre :**
1. Examiner les dimensions, les colonnes et quelques observations.
2. Décrire ce que représente une ligne.
3. Identifier les clés possibles et examiner les répétitions.
4. Contrôler les doublons exacts et distinguer répétitions normales et doublons possibles.

| Élément | Résultat / explication |
|---|---|
| Nombre de lignes / colonnes | 34 955 lignes × 6 colonnes |
| Contenu général, en une phrase | Pour chaque commune on a comme information son libellé et son type, le bassin de vie auquel elle appartient, son intitulé et son type, le nombre de communes qui se trouvent dans le bassin de vie ainsi que son département et sa région|
| Unité d’observation | Une commune identifiée par son code géographique |
| Identifiants / clés possibles | le code géographique |
| Une ou plusieurs lignes par objet ? Pourquoi ? |Une ligne par commune. |
| Doublons exacts : nombre / proportion | 0 (0 %) : aucune annonce identique sous deux identifiants différents|
| Répétitions métier : règle examinée / résultat | non applicable |

In [ ]:
codgeo_bv2012.head()

print(codgeo_bv2012.shape) 
display(codgeo_bv2012.head())
codgeo_bv2012.info() 


# La clé semble être le code géographique et on vérifie si elle est bien unique et jamais vide
print("code géographique unique :", codgeo_bv2012.index.is_unique)
print("code géographique vide   :", codgeo_bv2012.index.isna().sum())

# On regarde s'il y a des doublons exacts dans le dataset sans prendre en compte l'Index
codgeo_bv2012.duplicated().sum() 

In [ ]:
codgeo_bv2022.head()

In [ ]:
codgeo_bv2012.head()

## 3. Comprendre les variables

In [ ]:
codgeo_bv2012.info()

In [ ]:
codgeo_bv2012.nunique()

**À faire dans l’ordre :**
1. Identifier le sens et les unités des variables.
2. Distinguer nature statistique et type de stockage.
3. Examiner modalités et plages de valeurs pour vérifier cette compréhension.
4. Repérer les variables ambiguës ou constantes et leur utilité éventuelle.

| Variable | Signification | Nature statistique | unité | Type de stockage et cohérence | 
|---|---|---|---|---|
| LIBGEO | Libellé géographique | qualitative | N/A | str : cohérent |
| BV2012 | code géographique du bassin de vie 2012 | qualitative | N/A | str : cohérent |
| LIBBV2012 | libellé géographique du bassin de vie | qualitative | N/A | str : cohérent |
| DEP | Code géographique du département | qualitative | N/A | str :  cohérent car ce sont des nombres mais ils devront se comporter comme des modalités|
| REG | Code géographique de la région | qualitative | N/A | int64 : devront se comporter comme des modalités, passer en str? |
| NB_COM | Nombre de communes contenues dans le bassin de vie | quantitative continue | quantitative | int64 : cohérent |

In [ ]:
# Mesurer les NaN par variable 
manquants = pd.DataFrame({"nb_NaN": codgeo_bv2012.isna().sum(),
                          "pct_NaN": (codgeo_bv2012.isna().mean() * 100).round(1)})
manquants = manquants[manquants["nb_NaN"] > 0].sort_values("pct_NaN", ascending=False)
display(manquants)

Il n'y a aucune variable manquante dans tout le jeux de donnée des bassins de vie de 2012 à 2022.

# B. Bassins de vie entre 2022 et 2026

## 1. Situer la source et ouvrir les données

**À faire dans l’ordre :**
1. Identifier la source et consulter la documentation.
2. Préciser la collecte et le périmètre annoncés.
3. Charger les données et vérifier leur lecture.

| Élément | Information | Référence / réserve |
|---|---|---|
| Producteur / origine |INSEE | aucun fichier de métadonnées associé mais 2 feuilles descriptives dans l'excel|
| Finalité initiale | | |
| Mode de collecte | INSEE, code officiel géographique| |
| Extraction / mise à jour | 2022 à 2026 | |
| Documentation / licence | aucune information| |
| Population / objets annoncés | 1e feuille : nom de chaque bassin de vie, intitulé, type et nombre de communes présentes ; 2e feuilles : par code géographique officiel des communes on trouve son libellé, le bassin de vie auquel elle appartient, le type de commune, son département et sa région | |
| Période / territoire, si pertinents | France avec ses DOM-TOM| |

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
cd c:\Users\gabi9\Documents\GITHUB_PROJET_LIORA\projet_compagnon\data

In [ ]:
pip install openpyxl

In [63]:
bv2022=pd.read_excel('BV2022_2022_2026.xlsx', sheet_name='BV2022', skiprows=5)
compo_communale2022=pd.read_excel('BV2022_2022_2026.xlsx', sheet_name='Composition_communale', skiprows=5)

In [ ]:
bv2022.head()

In [ ]:
compo_communale2022.head()

In [ ]:
#Nombre de valeurs uniques par variable
print('valeurs uniques dans les compositions communales 2022 :\n', compo_communale2022.nunique())
print('valeurs uniques dans les bassins de vie 2022 :\n', bv2022.nunique())

In [ ]:
# Nombre d'annonces par département 
print(compo_communale2022["DEP"].value_counts())

print('nombre de communes totales de 2022 à 2026')
print(compo_communale2022["DEP"].value_counts().sum())

In [64]:
codgeo_bv2022=compo_communale2022.merge(bv2022, on=["BV2022", 'LIBBV2022'],how="left")
codgeo_bv2022.set_index('CODGEO', inplace=True)

In [ ]:
codgeo_bv2022.to_parquet('codgeo_bv2022.parquet', index=True)
codgeo_bv2022.to_csv('codgeo_bv2022.csv', index=True)

**Constat :**

Les codes géographiques officiels de chaque commune sont uniques, ils serviront d'index.
On retrouve le détail pour chacun des bassins de vie dans une autre feuille du tableur excel avec les informations le décrivant.

**Impact projet :**

Un fichier unique pour les codes géographique et les bases de vie est possible et essentiel pour la simplification de son utilisation.

**Action envisagée :**

Fusion des informations contenues dans les deux feuilles excel dans un unique DataFrame.

**Incertitudes / limites, si nécessaire :**

Un code géographique peut aussi être un code de bassin de vie. Viligance.

DONNEES DE 2022 A 2026

## 2. Comprendre la structure et une ligne

**À faire dans l’ordre :**
1. Examiner les dimensions, les colonnes et quelques observations.
2. Décrire ce que représente une ligne.
3. Identifier les clés possibles et examiner les répétitions.
4. Contrôler les doublons exacts et distinguer répétitions normales et doublons possibles.

| Élément | Résultat / explication |
|---|---|
| Nombre de lignes / colonnes | 34 935 lignes × 8 colonnes |
| Contenu général, en une phrase | Pour chaque commune on a comme information son libellé et son type, le bassin de vie auquel elle appartient, son intitulé et son type, le nombre de communes qui se trouvent dans le bassin de vie ainsi que son département et sa région|
| Unité d’observation | Une commune identifiée par son code géographique |
| Identifiants / clés possibles | le code géographique |
| Une ou plusieurs lignes par objet ? Pourquoi ? |Une ligne par commune. |
| Doublons exacts : nombre / proportion | 0 (0 %) : aucune annonce identique sous deux identifiants différents|
| Répétitions métier : règle examinée / résultat | non applicable |

In [ ]:
codgeo_bv2022.head()

In [ ]:
print(codgeo_bv2022.shape) 
display(codgeo_bv2022.head())
codgeo_bv2022.info() 


# La clé semble être le code géographique et on vérifie si elle est bien unique et jamais vide
print("code géographique unique :", codgeo_bv2022.index.is_unique)
print("code géographique vide   :", codgeo_bv2022.index.isna().sum())

# On regarde s'il y a des doublons exacts dans le dataset sans prendre en compte l'Index
codgeo_bv2022.duplicated().sum() 


**Constat :**

Le jeu de données compte 34 935 lignes × 8 colonnes. L'index, le code géographique est unique, sans manque ni répétitions.

**Impact projet :**

Ce jeux de données servira de lien entre toutes les autres.

**Action envisagée :**

/

**Incertitudes / limites, si nécessaire :**

/

## 3. Comprendre les variables

In [ ]:
codgeo_bv2022.info()

In [ ]:
codgeo_bv2022.nunique()

**À faire dans l’ordre :**
1. Identifier le sens et les unités des variables.
2. Distinguer nature statistique et type de stockage.
3. Examiner modalités et plages de valeurs pour vérifier cette compréhension.
4. Repérer les variables ambiguës ou constantes et leur utilité éventuelle.

| Variable | Signification | Nature statistique | unité | Type de stockage et cohérence | 
|---|---|---|---|---|
| LIBGEO | Libellé géographique | qualitative | N/A | str : cohérent |
| BV2012 | code géographique du bassin de vie 2012 | qualitative | N/A | str : cohérent |
| LIBBV2012 | libellé géographique du bassin de vie | qualitative | N/A | str : cohérent |
| DEP | Code géographique du département | qualitative | N/A| str :  cohérent car ce sont des nombres mais ils devront se comporter comme des modalités|
| REG | Code géographique de la région | qualitative | N/A  | int64 : devront se comporter comme des modalités, passer en str? |
| NB_COM | Nombre de communes contenues dans le bassin de vie | quantitative continue | quantitative | int64 : cohérent |
| TYPE_BV2022 | type de bassin de vie | qualitative |1-bassin issus d'un pôle partiel, dit 'rural', 2-bassin avec une seule commune pôle, dit 'intermédiare', 3- bassin avec plusieurs communes-pôles, dit 'dense' | int64 : modalités donc str? |
| TYPE_COMMUNE_BV2022 | catégorie de la commune vis-à-vis du bassin de vie | qualitative | 00-non pôle, 11- pôle partiel, 12- commune associée à un pôle partiel, 20-pôle | int64 : modalités donc str? |

In [ ]:
# Mesurer les NaN par variable 
manquants = pd.DataFrame({"nb_NaN": codgeo_bv2022.isna().sum(),
                          "pct_NaN": (codgeo_bv2022.isna().mean() * 100).round(1)})
manquants = manquants[manquants["nb_NaN"] > 0].sort_values("pct_NaN", ascending=False)
display(manquants)

Il n'y a aucune valeur manquante dans tout le jeux de données des bassins de vie de 2022 à 2026.

# Distribution des variables sur les bassins de vie

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

bv12 = pd.read_csv("codgeo_bv2012.csv", dtype=str)
bv22 = pd.read_csv("codgeo_bv2022.csv", dtype=str)
for d in (bv12, bv22):
    d["NB_COM"] = d["NB_COM"].astype(int)

# Un bassin = une ligne
b12 = bv12.drop_duplicates("BV2012")
b22 = bv22.drop_duplicates("BV2022")

# 1. Distribution du nombre de communes par bassin (2012 vs 2022)
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.logspace(0, np.log10(600), 40)
ax.hist(b12.NB_COM, bins=bins, alpha=.6, label="BV2012")
ax.hist(b22.NB_COM, bins=bins, alpha=.6, label="BV2022")
ax.set_xscale("log")
ax.set_xlabel("Nombre de communes par bassin (échelle log)")
ax.set_ylabel("Nombre de bassins")
ax.set_title("Taille des bassins de vie")
ax.legend()
plt.show()

Graphique : taille des bassins de vie.
Si on regarde la taille des bassins de vie c'est-à-dire le nombre de bassins en fonction du nombre de communes par bassin on peut voir que les données sont presque superposées. La répartition des différents bassins et communes est stable, d'autant plus qu'on observe un temps relativement long (2012 à 2026).
Il y a très peu de très gros bassins avec un nombre très importants de communes.

In [ ]:
# 2. Répartition des types de bassin 2022
labels = {"1": "Rural", "2": "Intermédiaire", "3": "Dense"}
t = b22.TYPE_BV2022.map(labels).value_counts()
t.plot.bar(title="Bassins 2022 par type", figsize=(6, 4))
plt.show()


Graphique : bassins de vie en 2022 classés par type.
Les bassins de type intermédiaire domine largement avec 1408 bassins soit 82% des bassins du fichier.
Le type dense ne regroupe que 271 bassins et le type de bassin rural seulement 28 bassins sont concernés.
Le type 3 urbain, malgré qu'il ne représente que 15.9% des bassins est aussi le plus étendu en terme de surface sur le territoire car chaque bassin possède en moyenne 28.8communes. On se représente plutôt bien ce type de bassins : une ville au centre très importante et sa couronne.

In [ ]:
# 3. Top 15 des plus grands bassins (en nb de communes)
top = b22.nlargest(15, "NB_COM")
sns.barplot(data=top, y="LIBBV2022", x="NB_COM", color="steelblue")
plt.title("15 plus grands bassins de vie (2022)")
plt.show()


Graphique : 15 plus grands bassins de vie.
Sans surprise Paris est le plus grand bassin de vie avec 440 communes et devance de loin le second qui est Reims (130 communes).
Le nombre de communes dans un bassin reflète le morcellement des petites communes notamment en zone rurale, qu'il pourra être intéressant d'analyser réellement en tant que bassin, contrairement à Paris où au sein d'un même bassin de vie les prix des biens peuvent énormément varier sur de plus courtes distances.

In [ ]:
# 4. Nombre de bassins par département / région
bv_dep = b22.groupby("DEP").size().sort_values(ascending=False).head(20)
bv_dep.plot.bar(figsize=(10, 4), title="Bassins par département (top 20)")
plt.show()

Graphique : top20 du nombre de bassins par département
On ne voit pas apparaître les bassins les plus denses car ils peuvent occuper un département voir plus comme c'est le cas de Paris : vigilance donc lors des agrégations.

In [ ]:
# 5. Comparaison 2012 → 2022 : communes qui changent de bassin
m = bv12.merge(bv22, on="CODGEO", suffixes=("_12", "_22"))
m["a_change"] = m.BV2012 != m.BV2022
chg = m.groupby("DEP_22").a_change.mean().mul(100).sort_values(ascending=False).head(20)
chg.plot.bar(figsize=(10, 4), title="% de communes ayant changé de bassin (top 20 départements)")
plt.ylabel("%")
plt.show()

Graphique : % de communes qui ont changé de bassins entre 2012 et 2026
25% des communes ont changé de code de bassin de vie.
La majorité concernent le littoral méditerranéen. Dans les Dom-Tom, Mayotte voit 100% de ses bassins remaniés.

In [ ]:
# 6. Flux de reclassement : Sankey ou matrice (exemple : taille 2012 vs 2022 du même centre)
tab = m.groupby(["LIBBV2012", "LIBBV2022"]).size().reset_index(name="n")
tab = tab[tab.LIBBV2012 != tab.LIBBV2022].nlargest(15, "n")
print(tab)

Quelques exemples de remaniements des bassins.

In [ ]:
# 7. Communes absentes d'un des deux millésimes
outer = bv12.merge(bv22, on="CODGEO", how="outer", indicator=True)
print(outer["_merge"].value_counts())

Entre les bassins de vie avant et après 2022, 24 communes ont disparu et 4 sont apparu.

# Conclusion sur l'usage des BV dans le projet

**Différenciation selon la temporalité**
Il sera nécessaire de découper ou attribuer différemment les codes de bassins de vie en fonction de si la vente a eu lieu avant 2022 (utilisation du codgeo_bv2012.csv) ou pendant/après 2022 (utilisation du codgeo_bv2022.csv).

**Limites :**
- les arrondissements des plus grosses villes ne sont pas précisés.
- évolution des bassins au cours du temps.
- si on cherche à utiliser les deux fichiers ont aura peut être générer un 'saut' suite aux changements sur certaines communes et bassins qui ne sera pas représentatif de la réalité.

**La commune, via son code géographique,** semble ici la clé principale pour permettre une jointure correcte car seulement très peu de mouvements (0.1% ont bougé, il faudra fusionner les anciennes communes avec celles qui les ont remplacées).

Les codes des **bassins de vie** ne changent pas mais leur périmètre peut changer drastiquement entre les deux versions.

**Utilisation des bassins de vie en lien avec le DVF+**
Idée principale : **garder toutes les informations**, merge chacun des jeux de données sur les bassins de vie en se basant sur le code géographique de chaque commune.
La majorité des analyses pourront porter sur **les bassins de vie en 2022** et éventuellement faire des tests comparatifs en les fichiers 2012 et 2022 pour voir si le découpage des bassins de vie a eu un impact sur certaines données.

Idée à discuter en groupe : **supprimer de l'analyse les 0.1% de communes qui changent** ? 

# Exploitation du fichier codgeo_bv2022.csv

Méthode choisie : conservation du fichier BV2022 car seulement 0.1% des communes sont impactées.

## Analyse lors du lien avec le fichier DVF+

**Test de la jointure entre la table cible DVF+ et la composition communale** (BV2022_2022_2026.xlsx, géographie 2026) :

• 6,4 % des ventes ne trouvent pas de bassin (613 973 sur 9,66 M)

• dont 582 153 pour les arrondissements de Paris, Lyon et Marseille (75101…, 69381…, 13201…) : le fichier Insee ne connaît que 75056, 69123 et 13055

• et 31 820 pour 1 666 anciens codes de communes fusionnées depuis 2014 (surtout 49, 50, 14)

**Proposition** : construire une table de correspondance unique pour toute l'équipe avec le code commune passé ou actuel et son bassin de vie 2022. Apparemment tu auras besoin de la table de passage de l'INSEE (Lien direct : table_passage_geo2003_geo2026.zip)
 Il reste juste à ajouter les arrondissements à la main (751xx → 75056, 6938x → 69123, 132xx → 13055), puis à joindre le bassin.

## Structure de la table de passage des communes

| Élément | Résultat / explication |
|---|---|
| Nombre de lignes / colonnes | 36 743 lignes × 3 colonnes |
| Contenu général, en une phrase | Pour chaque code géographique initial de 2003, correspondance avec le code géographique utilisé en 2026 |
| Unité d’observation | Une commune identifiée par son code géographique |
| Identifiants / clés possibles | le code géographique initial (CODGEO_INI) |
| Une ou plusieurs lignes par objet ? Pourquoi ? |Une ligne par commune, possible multiples lignes pour un code géographique initial car il y a eu des cas de scissions de communes |
| Doublons exacts : nombre / proportion | 0 (0 %) : pas de réel doublon car 2 lignes dans le cas d'une scission auront le même code géographique initial mais deux codes géographiques 'finaux' |
| Répétitions métier : règle examinée / résultat | non applicable |

In [ ]:
cd C:\Users\gabi9\Documents\GITHUB_PROJET_LIORA\projet_compagnon\data\

In [ ]:
import pandas as pd
import numpy as np

In [49]:
passage=pd.read_excel('table_passage_geo2003_geo2026.xlsx', sheet_name='Table de passage', skiprows=5)
scissions=pd.read_excel('table_passage_geo2003_geo2026.xlsx', sheet_name='Liste des scissions', skiprows=5)
fusions=pd.read_excel('table_passage_geo2003_geo2026.xlsx', sheet_name='Liste des fusions', skiprows=5)

In [ ]:
passage.head()

In [ ]:
print(passage.shape)

In [ ]:
passage.info()

## Comprendre les variables

| Variable | Signification | Nature statistique | unité | Type de stockage et cohérence | 
|---|---|---|---|---|
| CODGEO_INI | Code géographique de la commune en 2003 | qualitative | N/A | str : cohérent malgré que ça soit un nombre |
| CODGEO_2026 | code géographique de la commune en 2026 | qualitative | N/A | str : cohérent malgré que ça soit un nombre |
| LIBGEO_2026 | libellé géographique de la commune | qualitative | N/A | str : cohérent |

In [ ]:
print('valeurs uniques dans les compositions communales 2022 :\n', passage.nunique())

In [ ]:
print('valeurs uniques dans les compositions communales 2022 :\n', compo_communale2022.nunique())

In [ ]:
doublons = passage[passage.duplicated(subset=['CODGEO_INI'], keep=False)].sort_values('CODGEO_INI')
print(f"{len(doublons)} lignes en doublon sur 'CODGEO_INI'")
print(doublons)

La colonne CODGEO_INI pourrait servir de lien avec la table DVF+ car il convient à la fois les codes géographiques avant et après 2022. Il faut donc modifier le fichier codgeo_bv2022.csv qui devait servir de table unique de liaison entre les codes géographiques et les bassins de vie pour introduire la colonne CODEGEO_INI qui concernera les communes qui ont changé de codes.

**Problématique importante sur CODGEO_INI** sont les communes qui ont été scindées : pour un code géographique initial il y a plusieurs codes géographiques de sortie en 2026, quelle règle appliquer ?
Seconde problématique détectée lors du lien avec DVF+ : **absence de code géographique pour les arrondissements**. Il faudra donc les ajouter dans la table de passage.

## Modifications pour palier aux problématiques de la table de passage

### Ajout des arrondissements de Paris, Lyon et Marseille

In [ ]:
# code de la ville : (nom, codes des arrondissements)
villes = {
    "75056": ("Paris",     range(75101, 75121)),
    "69123": ("Lyon",      range(69381, 69390)),
    "13055": ("Marseille", range(13201, 13217)),
}

# Construction des nouvelles lignes
lignes = []
for code_ville, (nom, arrondissements) in villes.items():
    for n, code in enumerate(arrondissements, start=1):
        rang = "1er" if n == 1 else f"{n}e"
        lignes.append({
            "CODGEO_INI": str(code),
            "CODGEO_2026": code_ville,
            "LIBGEO_2026": f"{nom} {rang} Arrondissement",
        })

# Ajout au DataFrame contenant la table de passage
passage = pd.concat([passage, pd.DataFrame(lignes)], ignore_index=True)

In [ ]:
# observation des nouvelles lignes ajoutées pour Paris, Lyon et Marseille
print("Nouvelles lignes pour Paris :")
print(passage[passage["CODGEO_2026"] == "75056"])
print("\nNouvelles lignes pour Lyon :")
print(passage[passage["CODGEO_2026"] == "69123"])
print("\nNouvelles lignes pour Marseille :")
print(passage[passage["CODGEO_2026"] == "13055"])

### Gestion des communes scindées

Le DVF+ comporte les données de 2014 à 2025 donc toutes les modifications ayant eu lieu avant 2014 n'auront pas d'impact sur la lecture du DVF+.
On commence par jointre la table des passages à la table des compositions communales pour observer les cas de scission via le code géographique 2026.

In [ ]:
#fusion des tables des passages avec la composition communale 2022
passage_compo_communale2022 = passage.merge(compo_communale2022, left_on="CODGEO_2026", right_on="CODGEO", how="left").drop(columns='CODGEO')

In [39]:
passage_compo_communale2022.head()

,CODGEO_INI,CODGEO_2026,LIBGEO_2026,LIBGEO,BV2022,LIBBV2022,TYPE_COMMUNE_BV2022,DEP,REG
0,01001,01001,L'Abergement-Clémenciat,L'Abergement-Clémenciat,01093,Châtillon-sur-Chalaronne,0,01,84
1,01002,01002,L'Abergement-de-Varey,L'Abergement-de-Varey,01004,Ambérieu-en-Bugey,0,01,84
2,01004,01004,Ambérieu-en-Bugey,Ambérieu-en-Bugey,01004,Ambérieu-en-Bugey,20,01,84
3,01005,01005,Ambérieux-en-Dombes,Ambérieux-en-Dombes,01333,Saint-André-de-Corcy,0,01,84
4,01006,01006,Ambléon,Ambléon,01034,Belley,0,01,84


Nous allons ensuite observer les doublons de codgeo_2026 et codgeo_ini pour voir leurs effets sur le bassins de vie qui découle d'une modification de code géographique.

In [45]:
# 1. Doublons de CODGEO_INI : les communes qui ont été scindées
dbl_ini = passage_compo_communale2022[passage_compo_communale2022.duplicated("CODGEO_INI", keep=False)].sort_values("CODGEO_INI")
dbl_ini_bv_idem = dbl_ini[dbl_ini.duplicated('BV2022', keep=False)]
print(f"{len(dbl_ini_bv_idem)} lignes en doublon sur CODGEO_INI et sur BV2022 ({dbl_ini_bv_idem['CODGEO_INI'].nunique()} codes)")
print(dbl_ini_bv_idem)

48 lignes en doublon sur CODGEO_INI et sur BV2022 (22 codes)
      CODGEO_INI CODGEO_2026              LIBGEO_2026  \
4657       14149       14149         Cesny-aux-Vignes   
4965       14149       14482                    Ouézy   
5178       14712       14712                   Troarn   
36736      14712       14666              Sannerville   
5364       15141       15141      Neussargues-Moissac   
36737      15141       15031                   Celles   
36738      15141       15035             Chalinargues   
36739      15141       15047                Chavagnac   
36740      15141       15171         Sainte-Anastasie   
7817       22179       22179                   Fréhel   
7839       22179       22201                 Plévenon   
12352      31483       31300                   Lieoux   
12353      31483       31483            Saint-Gaudens   
13936      35130       35130            Hédé-Bazouges   
14119      35130       35317         Saint-Symphorien   
18024      47157       4715

In [48]:
differences=dbl_ini[~dbl_ini['CODGEO_2026'].isin(dbl_ini_bv_idem['CODGEO_2026'])]
print(differences)

      CODGEO_INI CODGEO_2026         LIBGEO_2026              LIBGEO BV2022  \
20299      52427       52266   Laneuville-à-Rémy   Laneuville-à-Rémy  52550   
21313      55419       55068  Brabant-en-Argonne  Brabant-en-Argonne  51507   
31472      76676       76601        Saint-Lucien        Saint-Lucien  76312   
31473      76676       76676        Sigy-en-Bray        Sigy-en-Bray  76276   

              LIBBV2022  TYPE_COMMUNE_BV2022 DEP  REG  
20299             Wassy                    0  52   44  
21313  Sainte-Menehould                    0  55   44  
31472   Gournay-en-Bray                    0  76   28  
31473   Forges-les-Eaux                    0  76   28  


Uniquement 4 communes dont le code géographique initial est identique changent de bassins de vie : les communes de codes 2026 52266, 55068, 76601, 76676.
On observe la liste des scissions pour voir les dates de scission et si le DVF+ sera impacté.

In [ ]:
scissions['COM_FIN']=scissions['COM_FIN'].astype(str)

In [60]:
scissions.loc[scissions['COM_FIN'].isin(['52266', '55068', '76601', '76676'])]

,ANNEE_MODIF,COM_INI,COM_FIN,LIB_COM_INI,LIB_COM_FIN
12,2017,76676,76601,Sigy-en-Bray,Saint-Lucien
13,2017,76676,76676,Sigy-en-Bray,Sigy-en-Bray
21,2012,52427,52266,Robert-Magny-Laneuville-à-Rémy,Laneuville-à-Rémy
47,2004,55419,55068,Récicourt,Brabant-en-Argonne


2 des 4 scissions qui pourraient poser problèmes ont eu lieu avant les dates contenues dans le DVF+. 2 scissions en 2017 ont fait changé la commune de bassins de vie : le type de bassins de vie est identique mais pas le nombre de communes inclues dans le bassins de vie.

In [66]:
codgeo_bv2022[codgeo_bv2022.index.isin(['76676', '76601'])]

,LIBGEO,BV2022,LIBBV2022,TYPE_COMMUNE_BV2022,DEP,REG,TYPE_BV2022,NB_COM
CODGEO,,,,,,,,
76601,Saint-Lucien,76312,Gournay-en-Bray,0,76,28,2,35
76676,Sigy-en-Bray,76276,Forges-les-Eaux,0,76,28,2,28


In [69]:
# 2. Doublons de CODGEO_2026 : communes qui ont été fusionnées
dbl_2026 = passage_compo_communale2022[passage_compo_communale2022.duplicated("CODGEO_2026", keep=False)].sort_values(["CODGEO_2026", "CODGEO_INI"])
dbl_2026_bv_idem = dbl_2026[dbl_2026.duplicated('BV2022', keep=False)]
print(f"\n{len(dbl_2026_bv_idem)} lignes en doublon sur CODGEO_2026 avec BV2022 identique:")
print(dbl_2026_bv_idem)


2794 lignes en doublon sur CODGEO_2026 avec BV2022 identique:
      CODGEO_INI CODGEO_2026      LIBGEO_2026           LIBGEO BV2022  \
13         01015       01015  Arboys en Bugey  Arboys en Bugey  01034   
307        01340       01015  Arboys en Bugey  Arboys en Bugey  01034   
21         01025       01025   Bâgé-Dommartin   Bâgé-Dommartin  71270   
131        01144       01025   Bâgé-Dommartin   Bâgé-Dommartin  71270   
29         01033       01033       Valserhône       Valserhône  01033   
...          ...         ...              ...              ...    ...   
36343      93066       93066      Saint-Denis      Saint-Denis  75056   
36411      95040       95040          Avernes          Avernes  75056   
36473      95259       95040          Avernes          Avernes  75056   
36447      95169       95169          Commeny          Commeny  95355   
36479      95282       95169          Commeny          Commeny  95355   

            LIBBV2022  TYPE_COMMUNE_BV2022 DEP  REG  
13    

In [72]:
dbl_2026.shape

(2794, 9)

In [74]:
dbl_2026_bv_idem.shape

(2794, 9)

Parmi les fusions de communes (doublons de codes géographiques en 2026 avec des codes initiaux différents), aucune commune ne change de bassins de vie.
Ce seront des cas simples à traiter lorsque nous utiliserons les bassins de vie pour lire le DVF+, que l'on considère les codes géographiques initiaux ou de 2026.

### Etablissement d'une nouvelle base de donnée qui servira de clé de liaison

Il faut maintenant merge avec les informations relatives à chacun des bassins de vie pour avoir une clé avec toutes les informations.

In [75]:
passage_compo_communale2022.head()

,CODGEO_INI,CODGEO_2026,LIBGEO_2026,LIBGEO,BV2022,LIBBV2022,TYPE_COMMUNE_BV2022,DEP,REG
0,01001,01001,L'Abergement-Clémenciat,L'Abergement-Clémenciat,01093,Châtillon-sur-Chalaronne,0,01,84
1,01002,01002,L'Abergement-de-Varey,L'Abergement-de-Varey,01004,Ambérieu-en-Bugey,0,01,84
2,01004,01004,Ambérieu-en-Bugey,Ambérieu-en-Bugey,01004,Ambérieu-en-Bugey,20,01,84
3,01005,01005,Ambérieux-en-Dombes,Ambérieux-en-Dombes,01333,Saint-André-de-Corcy,0,01,84
4,01006,01006,Ambléon,Ambléon,01034,Belley,0,01,84


In [76]:
bv2022.head()

,BV2022,LIBBV2022,TYPE_BV2022,NB_COM
0,01004,Ambérieu-en-Bugey,2,34
1,01033,Valserhône,2,20
2,01034,Belley,2,35
3,01053,Bourg-en-Bresse,3,23
4,01072,Ceyzériat,2,16


In [80]:
bv_liaison_2022 = passage_compo_communale2022.merge(bv2022, left_on=["BV2022", "LIBBV2022"], right_on=["BV2022", "LIBBV2022"], how="left").drop(columns='LIBGEO')

In [81]:
bv_liaison_2022.head()

,CODGEO_INI,CODGEO_2026,LIBGEO_2026,BV2022,LIBBV2022,TYPE_COMMUNE_BV2022,DEP,REG,TYPE_BV2022,NB_COM
0,01001,01001,L'Abergement-Clémenciat,01093,Châtillon-sur-Chalaronne,0,01,84,2,17
1,01002,01002,L'Abergement-de-Varey,01004,Ambérieu-en-Bugey,0,01,84,2,34
2,01004,01004,Ambérieu-en-Bugey,01004,Ambérieu-en-Bugey,20,01,84,2,34
3,01005,01005,Ambérieux-en-Dombes,01333,Saint-André-de-Corcy,0,01,84,2,7
4,01006,01006,Ambléon,01034,Belley,0,01,84,2,35


In [82]:
bv_liaison_2022.to_parquet('bv_liaison_2022.parquet', index=True)
bv_liaison_2022.to_csv('bv_liaison_2022.csv', index=True)

**COLONNE A UTILISER POUR LIRE LE DVF+** 
Il faudra privilégier la colonne CODGEO_INI pour être sûr de tout prendre en compte.